# 04 — Long-Document Question Answering with Longformer

This notebook walks through the production modules used by the Gradio/Hugging
Face Space. It does not duplicate the application code.

> **Responsible use:** Use only public, synthetic, or non-sensitive documents.
> Model answers and confidence proxies require human review.


## 1. Environment

Run this notebook from the project folder after installing `requirements.txt`.
The first inference downloads the published Longformer checkpoint.


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import InferenceConfig
from src.data_preprocessing import attach_document_text, load_qa_pairs
from src.document_loader import load_sample_document
from src.inference_pipeline import LongDocumentQAPipeline
from src.model_evaluation import evaluate_dataframe, summarize_evaluation


## 2. Runtime configuration

The checkpoint supports approximately 4,096 tokens. The demo defaults to a
2,048-token runtime window to reduce CPU latency.


In [ ]:
config = InferenceConfig()
config.to_dict()


## 3. Load a safe sample document


In [ ]:
document = load_sample_document("quality_capa_report.txt", config)
print(document.source_name)
print(document.word_count, "words")
print(document.text[:1000])


## 4. Run Longformer QA

The model is lazy-loaded in this cell. Documents longer than the selected
runtime window are encoded as overlapping token windows.


In [ ]:
pipeline = LongDocumentQAPipeline(config)

result = pipeline.answer(
    question="Who was assigned as the CAPA owner?",
    document_text=document.text,
    source_name=document.source_name,
    max_length=2048,
    stride=256,
)
result.to_dict()


## 5. Inspect evidence and diagnostics


In [ ]:
print("Answer:", result.answer)
print("Confidence proxy:", result.confidence_proxy)
print("Supporting paragraph:", result.supporting_paragraph)
print("Paragraph index:", result.paragraph_index)
print("Windows:", result.window_count)
print("Latency:", result.latency_seconds)
print("Warnings:", result.warnings)


## 6. Evaluate the sample QA pairs

The following cell computes actual Exact Match, token-level F1, evidence recall,
latency, and per-example observations. Do not publish metrics until it finishes.


In [ ]:
qa_pairs = load_qa_pairs(PROJECT_ROOT / "data" / "sample_qa_pairs.csv")
qa_pairs = attach_document_text(
    qa_pairs,
    PROJECT_ROOT / "data" / "sample_documents",
)
evaluation = evaluate_dataframe(
    pipeline,
    qa_pairs,
    max_length=2048,
    stride=256,
)
evaluation[[
    "example_id",
    "question",
    "reference_answer",
    "predicted_answer",
    "exact_match",
    "token_f1",
    "evidence_recall",
    "confidence_proxy",
    "latency_seconds",
]]


In [ ]:
summarize_evaluation(evaluation)


## 7. Save reproducible outputs


In [ ]:
from src.model_evaluation import save_evaluation_outputs, write_manual_error_analysis

output_dir = PROJECT_ROOT / "outputs"
summary = save_evaluation_outputs(evaluation, output_dir)
write_manual_error_analysis(evaluation, output_dir / "manual_error_analysis.md")
summary


## 8. Launch the Gradio application

Run this command in a terminal from the project directory:

```bash
python app.py
```
